In [0]:
# =============================================================
# CELDA 1: SETUP — Credenciales, Imports, Config, Batch ID
# =============================================================
import boto3
import json
import re
import unicodedata
import uuid
from datetime import datetime, timezone
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import IntegerType, LongType, StringType, StructField, StructType, TimestampType

# --- 1. Batch ID único por ejecución (trazabilidad MLOps) ---
BATCH_ID = str(uuid.uuid4())
BATCH_TIMESTAMP = datetime.now(timezone.utc).isoformat()
print(f"🔑 Batch ID: {BATCH_ID}")
print(f"🕐 Timestamp: {BATCH_TIMESTAMP}")

# --- 2. Cargar Credenciales ---
# Prioridad: Databricks secrets (producción) → archivo local (desarrollo)
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales cargadas desde Databricks Secrets.")
except Exception:
    try:
        with open("aws_secrets.json", "r") as f:
            config = json.load(f)
        print("✅ Credenciales cargadas desde aws_secrets.json (local).")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles. Configura Databricks Secrets (scope='aws', keys='access_key','secret_key') o coloca aws_secrets.json.")
    except json.JSONDecodeError:
        raise SystemExit("❌ aws_secrets.json tiene formato inválido.")

# --- 3. Cliente Boto3 ---
REGION = "us-east-1"
BUCKET = "bronce-scrap-date"

s3_client = boto3.client(
    "s3",
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
    region_name=REGION,
)

# --- 4. Config Spark S3 (reutilizable) ---
S3_OPTIONS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

print(f"✅ Bucket: {BUCKET} | Region: {REGION}")

In [0]:
# =============================================================
# CELDA 2: FUNCIONES — Descubrimiento + Procesamiento Idempotente
# =============================================================

FUENTES_IGNORADAS_RAW = {
    "archive", "checkpoint", "checkpoints", "control", "metadata",
    "ops", "rpa", "state", "states", "status",
}
RAW_SOURCE_CHECKPOINT_SUBPATHS = [
    "checkpoints/",
    "checkpoint/",
    "metadata/checkpoints/",
    "metadata/checkpoint/",
    "control/checkpoints/",
    "control/checkpoint/",
    "ops/checkpoints/",
    "ops/checkpoint/",
    "rpa/checkpoints/",
    "rpa/checkpoint/",
    "state/",
    "states/",
    "status/",
]
RAW_CHECKPOINT_CONTAINER_TOKENS = FUENTES_IGNORADAS_RAW.union({"raw", "bronze"})
RAW_CHECKPOINT_NON_PORTAL_FILENAMES = {
    "checkpoint", "current", "estado", "latest", "metadata", "progress", "state", "status",
}
CHECKPOINT_BRONZE_PATH = f"s3a://{BUCKET}/bronze/portal_checkpoints_raw/"

def obtener_fuentes_disponibles(bucket):
    """Escanea carpetas en raw/ usando Delimiter='/' de S3."""
    print("📡 Escaneando carpetas en 'raw/'...")
    response = s3_client.list_objects_v2(Bucket=bucket, Prefix="raw/", Delimiter="/")

    fuentes = []
    if "CommonPrefixes" in response:
        for prefix in response["CommonPrefixes"]:
            nombre = prefix["Prefix"].replace("raw/", "").strip("/")
            if nombre:
                fuentes.append(nombre)

    print(f"✅ {len(fuentes)} fuentes encontradas: {fuentes}")
    return fuentes


def listar_parquets(bucket, fuente):
    """Lista TODOS los .parquet en raw/{fuente}/ (excluye archive/)."""
    prefix = f"raw/{fuente}/"
    paginator = s3_client.get_paginator("list_objects_v2")
    archivos = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]
                if key.endswith(".parquet") and "/archive/" not in key:
                    archivos.append(key)

    return archivos


def resumir_parquets_pendientes(bucket, fuentes):
    """Genera visibilidad temprana del backlog raw antes de escribir a Bronze."""
    resumen = []
    for fuente in fuentes:
        archivos = listar_parquets(bucket, fuente)
        resumen.append({
            "fuente": fuente,
            "parquets_pendientes": len(archivos),
            "primer_archivo": archivos[0].split("/")[-1] if archivos else None,
        })
    return resumen


def archivar_archivo(bucket, key_origen, fuente):
    """Mueve un archivo de batches/ a archive/ (copy + delete)."""
    nombre_archivo = key_origen.split("/")[-1]
    key_destino = f"raw/{fuente}/archive/{nombre_archivo}"

    s3_client.copy_object(
        Bucket=bucket,
        CopySource={"Bucket": bucket, "Key": key_origen},
        Key=key_destino,
    )
    s3_client.delete_object(Bucket=bucket, Key=key_origen)
    return key_destino


def normalize_checkpoint_text(value):
    text = "" if value is None else str(value)
    text = text.lower().strip()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-z0-9\s/_-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_checkpoint_datetime(raw_value):
    if not raw_value:
        return None
    try:
        parsed = datetime.fromisoformat(str(raw_value).replace("Z", "+00:00"))
        if parsed.tzinfo is None:
            parsed = parsed.replace(tzinfo=timezone.utc)
        return parsed
    except Exception:
        return None


def infer_portal_from_checkpoint_key(key):
    parts = [normalize_checkpoint_text(part) for part in str(key).split("/") if part]
    if not parts:
        return ""

    filename = normalize_checkpoint_text(parts[-1].replace(".json", ""))
    if (
        filename
        and filename not in RAW_CHECKPOINT_NON_PORTAL_FILENAMES
        and "checkpoint" not in filename
    ):
        return filename

    for part in reversed(parts[:-1]):
        if (
            part
            and part not in RAW_CHECKPOINT_CONTAINER_TOKENS
            and "checkpoint" not in part
        ):
            return part

    return filename


def list_checkpoint_json_keys(bucket, fuentes):
    keys = []
    scanned_prefixes = []
    seen = set()

    candidate_prefixes = ["raw/checkpoints/"]
    for fuente in fuentes:
        candidate_prefixes.append(f"raw/{fuente}/")
        for subpath in RAW_SOURCE_CHECKPOINT_SUBPATHS:
            candidate_prefixes.append(f"raw/{fuente}/{subpath}")

    paginator = s3_client.get_paginator("list_objects_v2")
    for prefix in dict.fromkeys(candidate_prefixes):
        try:
            prefix_has_content = False
            for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
                for obj in page.get("Contents", []):
                    key = obj.get("Key", "")
                    key_lower = key.lower()
                    prefix_has_content = True
                    if not key_lower.endswith(".json") or "/archive/" in key_lower:
                        continue
                    if "checkpoint" not in key_lower and not key_lower.endswith("/state.json") and not key_lower.endswith("/status.json"):
                        continue
                    if key in seen:
                        continue
                    seen.add(key)
                    keys.append(key)
            if prefix_has_content:
                scanned_prefixes.append(prefix)
        except Exception:
            continue

    return keys, list(dict.fromkeys(scanned_prefixes))


def procesar_checkpoints_raw(bucket, fuentes):
    """Materializa snapshot técnico raw -> Bronze; limpieza analítica ocurre después en Silver."""
    checkpoint_keys, scanned_prefixes = list_checkpoint_json_keys(bucket, fuentes)
    rows = []

    for key in checkpoint_keys:
        portal_name = infer_portal_from_checkpoint_key(key)
        try:
            payload = s3_client.get_object(Bucket=bucket, Key=key)["Body"].read()
            data = json.loads(payload)
            rows.append({
                "portal": normalize_checkpoint_text(data.get("portal") or portal_name),
                "checkpoint_key": key,
                "checkpoint_source_path": key.rsplit("/", 1)[0] + "/",
                "checkpoint_activo": 1,
                "checkpoint_last_page": int(data.get("last_page")) if data.get("last_page") is not None else None,
                "checkpoint_total_scraped": int(data.get("total_scraped")) if data.get("total_scraped") is not None else None,
                "checkpoint_updated_at": parse_checkpoint_datetime(data.get("updated_at")),
                "checkpoint_updated_at_raw": str(data.get("updated_at")) if data.get("updated_at") is not None else None,
                "checkpoint_read_error": None,
            })
        except Exception as exc:
            rows.append({
                "portal": portal_name,
                "checkpoint_key": key,
                "checkpoint_source_path": key.rsplit("/", 1)[0] + "/",
                "checkpoint_activo": 1,
                "checkpoint_last_page": None,
                "checkpoint_total_scraped": None,
                "checkpoint_updated_at": None,
                "checkpoint_updated_at_raw": None,
                "checkpoint_read_error": str(exc)[:200],
            })

    checkpoint_schema = StructType([
        StructField("portal", StringType(), True),
        StructField("checkpoint_key", StringType(), True),
        StructField("checkpoint_source_path", StringType(), True),
        StructField("checkpoint_activo", IntegerType(), True),
        StructField("checkpoint_last_page", LongType(), True),
        StructField("checkpoint_total_scraped", LongType(), True),
        StructField("checkpoint_updated_at", TimestampType(), True),
        StructField("checkpoint_updated_at_raw", StringType(), True),
        StructField("checkpoint_read_error", StringType(), True),
    ])

    df_checkpoints = spark.createDataFrame(rows, schema=checkpoint_schema) if rows else spark.createDataFrame([], schema=checkpoint_schema)
    df_checkpoints = (
        df_checkpoints
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("batch_id", lit(BATCH_ID))
    )

    writer = df_checkpoints.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    for k, v in S3_OPTIONS.items():
        writer = writer.option(k, v)
    writer.save(CHECKPOINT_BRONZE_PATH)

    print(f"🛰️ Checkpoints raw procesados: {len(rows)} filas → {CHECKPOINT_BRONZE_PATH}")
    print("   Nota: esta tabla es técnica; la consolidación usable ocurre en Silver.")
    if scanned_prefixes:
        print(f"   Prefijos explorados con contenido: {scanned_prefixes}")
    else:
        print("   No se detectaron prefijos con checkpoints en raw/.")

    return {
        "rows": len(rows),
        "prefixes": scanned_prefixes,
        "path": CHECKPOINT_BRONZE_PATH,
    }


def procesar_fuente(fuente, bucket):
    """
    Procesa todos los parquets de una fuente con idempotencia:
    1. Lee cada parquet con Spark
    2. Enriquece con metadata MLOps
    3. Escribe a Delta Bronze (append + mergeSchema)
    4. Mueve archivo a archive/ SOLO si write exitoso
    """
    archivos = listar_parquets(bucket, fuente)

    if not archivos:
        print(f"  ⏩ {fuente}: Sin archivos .parquet nuevos")
        return {"ok": 0, "error": 0, "archivados": 0, "pending": 0}

    print(f"\n🚀 {fuente.upper()}: {len(archivos)} archivo(s) a procesar")

    contadores = {"ok": 0, "error": 0, "archivados": 0, "pending": len(archivos)}
    ruta_bronze = f"s3a://{bucket}/bronze/{fuente}/"

    for key in archivos:
        nombre_archivo = key.split("/")[-1]
        ruta_s3a = f"s3a://{bucket}/{key}"

        # --- PASO 1: Leer parquet ---
        try:
            df_raw = spark.read.format("parquet")
            for k, v in S3_OPTIONS.items():
                df_raw = df_raw.option(k, v)
            df_raw = df_raw.load(ruta_s3a)
        except Exception as e:
            print(f"  ❌ ERROR lectura [{nombre_archivo}]: {e}")
            contadores["error"] += 1
            continue

        # --- PASO 2: Enriquecer con metadata ---
        df_enriched = (
            df_raw
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_file", lit(nombre_archivo))
            .withColumn("batch_id", lit(BATCH_ID))
            .withColumn("source_system", lit(fuente))
        )

        # --- PASO 3: Escribir a Delta Bronze ---
        try:
            writer = df_enriched.write.format("delta").mode("append").option("mergeSchema", "true")
            for k, v in S3_OPTIONS.items():
                writer = writer.option(k, v)
            writer.save(ruta_bronze)

            contadores["ok"] += 1
            print(f"  ✅ Guardado: {nombre_archivo} ({df_raw.count()} filas)")
        except Exception as e:
            print(f"  ❌ ERROR escritura [{nombre_archivo}]: {e}")
            contadores["error"] += 1
            continue  # NO mover → reintento en próxima ejecución

        # --- PASO 4: Archivar (SOLO si write exitoso) ---
        try:
            archivar_archivo(bucket, key, fuente)
            contadores["archivados"] += 1
        except Exception as e:
            print(f"  ⚠️ WARN guardado en Bronze pero no archivado [{nombre_archivo}]: {e}")

    print(f"  📊 {fuente}: {contadores['ok']} OK | {contadores['error']} error | {contadores['archivados']} archivados")
    return contadores

In [0]:
# =============================================================
# CELDA 3: EJECUCIÓN — Orquestación + Resumen Global
# =============================================================

fuentes_detectadas = obtener_fuentes_disponibles(BUCKET)
fuentes_a_procesar = [fuente for fuente in fuentes_detectadas if fuente not in FUENTES_IGNORADAS_RAW]
resumen_pendientes = resumir_parquets_pendientes(BUCKET, fuentes_a_procesar)
fuentes_con_parquets = [item["fuente"] for item in resumen_pendientes if item["parquets_pendientes"] > 0]

print("\n🧭 PRE-CHEQUEO RAW → BRONZE")
print(f"   Fuentes útiles detectadas: {len(fuentes_a_procesar)}")
print(f"   Fuentes con parquets pendientes: {len(fuentes_con_parquets)}")
for item in resumen_pendientes:
    print(
        f"   - {item['fuente']}: {item['parquets_pendientes']} parquet(s) pendientes"
        + (f" | primero={item['primer_archivo']}" if item["primer_archivo"] else "")
    )

checkpoint_result = procesar_checkpoints_raw(BUCKET, fuentes_a_procesar)

resumen_global = {"total_ok": 0, "total_error": 0, "total_archivados": 0, "total_pendientes": 0}

for fuente in fuentes_a_procesar:
    resultado = procesar_fuente(fuente, BUCKET)
    resumen_global["total_ok"] += resultado["ok"]
    resumen_global["total_error"] += resultado["error"]
    resumen_global["total_archivados"] += resultado["archivados"]
    resumen_global["total_pendientes"] += resultado["pending"]

# --- Resumen Final ---
print("\n" + "=" * 60)
print(f"🏁 RESUMEN BATCH {BATCH_ID[:8]}...")
print(f"   Fuentes detectadas en raw/: {len(fuentes_detectadas)}")
print(f"   Fuentes parquet evaluadas: {len(fuentes_a_procesar)}")
print(f"   Total parquet pendientes al inicio: {resumen_global['total_pendientes']}")
print(f"   Checkpoints raw materializados: {checkpoint_result['rows']}")
print(f"   Ruta técnica checkpoints Bronze: {checkpoint_result['path']}")
print(f"   Archivos guardados en Bronze: {resumen_global['total_ok']}")
print(f"   Archivos archivados (movidos): {resumen_global['total_archivados']}")
print(f"   Archivos con error (quedan en raw/): {resumen_global['total_error']}")
print("   Siguiente paso esperado: ejecutar Silver para normalizar y filtrar antes de Gold/Modelo.")
print("=" * 60)